In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set the path to your shared folder
# This works for everyone because of the shortcut they made in Step 1!
project_path = '/content/drive/MyDrive/RU_AI_Campus_Team'

# 3. Move into the folder
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"Success! Working in: {os.getcwd()}")
else:
    print("Error: Could not find the project folder. Did you add the shortcut to your My Drive?")

In [ ]:
import pandas as pd

# Path to your SAMHSA file in Google Drive
file_path = "SAMHSA_2/National Directory MH 2024_Final.xlsx"

# Check sheet names
xls = pd.ExcelFile(file_path)
print("Sheets:", xls.sheet_names)

# Load the first sheet (adjust if needed)
df_samhsa = pd.read_excel(file_path, sheet_name=xls.sheet_names[0])

# Preview
print(df_samhsa.shape)
df_samhsa.head()

In [ ]:
df_samhsa.info()

In [ ]:
df_samhsa['state'].unique()

In [ ]:
# selected_states = ['NJ', 'NY', 'PA', 'DE']
# df_samhsa = df_samhsa[df_samhsa['state'].isin(selected_states)]

print(f"Filtered DataFrame shape: {df_samhsa.shape}")

In [ ]:
# Create one facility name from name 1 and name 2
df_samhsa['facility_name'] = df_samhsa['name1'] + ' ' + df_samhsa['name2'].fillna('')
df_samhsa['facility_name'] = df_samhsa['facility_name'].str.strip()

df_samhsa.drop(['name1', 'name2'], axis=1, inplace=True)

In [ ]:
# Replace '---' with empty string and fill NaN with empty string for street columns
df_samhsa['street1'] = df_samhsa['street1'].replace('---', '').fillna('')
df_samhsa['street2'] = df_samhsa['street2'].replace('---', '').fillna('')

# Combine street1 and street2 into a new 'address' column
df_samhsa['address'] = df_samhsa['street1'] + ' ' + df_samhsa['street2']

# Clean up any extra spaces that might result from combining
df_samhsa['address'] = df_samhsa['address'].str.strip()

# Display a sample to verify
display(df_samhsa[['street1', 'street2', 'address']].head())

df_samhsa.drop(['street1', 'street2'], axis=1, inplace=True)

In [ ]:
# Standardize 'state' to uppercase
df_samhsa['state'] = df_samhsa['state'].str.upper()

# Convert 'zip' to a 5-digit string, padding with leading zeros
df_samhsa['zip'] = df_samhsa['zip'].astype(str).str.zfill(5)

# Display a sample to verify
display(df_samhsa[['state', 'zip']].head())

In [ ]:
import re

# ---------- CODE MAPS (based on the SAMHSA PDF key) ----------
service_maps = {
    "type_of_care": {
        "SA": "Substance use treatment",
        "MH": "Mental health treatment",
        "SUMH": "Treatment for co-occurring substance use plus mental health conditions",
    },
    "service_setting": {
        "HI": "Hospital inpatient / 24-hour hospital inpatient",
        "OP": "Outpatient",
        "PHDT": "Partial hospitalization / day treatment",
        "RES": "Residential / 24-hour residential",
    },
    "facility_type": {
        "CMHC": "Community mental health center",
        "CBHC": "Certified Community Behavioral Health Clinic",
        "MSMH": "Multi-setting mental health facility",
        "OMH": "Outpatient mental health facility",
        "ORES": "Other residential treatment facility",
        "PH": "Partial hospitalization / day treatment",
        "PSY": "Psychiatric hospital",
        "RTCA": "Residential treatment center for adults",
        "RTCC": "Residential treatment center for children",
        "IPSY": "Separate inpatient psychiatric unit of a general hospital",
        "SHP": "State hospital",
        "VAHC": "Veterans Affairs Medical Center or other VA healthcare facility",
    },
    "pharmacotherapies": {
        "CHLOR": "Chlorpromazine",
        "DROPE": "Droperidol",
        "FLUPH": "Fluphenazine",
        "HALOP": "Haloperidol",
        "LOXAP": "Loxapine",
        "PERPH": "Perphenazine",
        "PIMOZ": "Pimozide",
        "PROCH": "Prochlorperazine",
        "THIOT": "Thiothixene",
        "THIOR": "Thioridazine",
        "TRIFL": "Trifluoperazine",
        "ARIPI": "Aripiprazole",
        "ASENA": "Asenapine",
        "BREXP": "Brexpiprazole",
        "CARIP": "Cariprazine",
        "CLOZA": "Clozapine",
        "ILOPE": "Iloperidone",
        "LURAS": "Lurasidone",
        "OLANZ": "Olanzapine",
        "OLANZF": "Olanzapine/Fluoxetine combination",
        "PALIP": "Paliperidone",
        "QUETI": "Quetiapine",
        "RISPE": "Risperidone",
        "ZIPRA": "Ziprasidone",
        "NRT": "Nicotine replacement",
        "NSC": "Non-nicotine smoking/tobacco cessation",
        "ANTPYCH": "Antipsychotics used in treatment of serious mental illness",
    },
    "treatment_approaches": {
        "AT": "Activity therapy",
        "CBT": "Cognitive behavioral therapy",
        "CRT": "Cognitive remediation therapy",
        "CFT": "Couples/family therapy",
        "DBT": "Dialectical behavior therapy",
        "ECT": "Electroconvulsive therapy",
        "EMDR": "Eye Movement Desensitization and Reprocessing therapy",
        "GT": "Group therapy",
        "IDD": "Integrated Mental and Substance Use Disorder treatment",
        "IPT": "Individual psychotherapy",
        "KIT": "Ketamine infusion therapy",
        "TMS": "Transcranial Magnetic Stimulation",
        "TELE": "Telemedicine / telehealth therapy",
        "AIM": "Abnormal involuntary movement scale",
    },
    "emergency_services": {
        "CIT": "Crisis intervention team",
        "PEON": "Psychiatric emergency onsite services",
        "PEOFF": "Psychiatric emergency mobile/off-site services",
        "WI": "Psychiatric emergency walk-in services",
    },
    "facility_operation": {
        "LCCG": "Local, county, or community government",
        "DDF": "Department of Defense",
        "IH": "Indian Health Services",
        "PVTP": "Private for-profit organization",
        "PVTN": "Private non-profit organization",
        "STG": "State government",
        "TBG": "Tribal government",
        "VAMC": "U.S. Department of Veterans Affairs",
        "FED": "Federal Government",
    },
    "license_certification": {
        "FQHC": "Federally Qualified Health Center",
        "MHC": "Mental health clinic or mental health center",
    },
    "payment_funding": {
        "CLF": "County or local government funds",
        "CMHG": "Community Mental Health Block Grants",
        "CSBG": "Community Service Block Grants",
        "FG": "Federal Grants",
        "ITU": "IHS/Tribal/Urban funds",
        "MC": "Medicare",
        "MD": "Medicaid",
        "MI": "Federal military insurance",
        "OSF": "Other State funds",
        "PI": "Private health insurance",
        "PCF": "Private or Community foundation",
        "SCJJ": "State corrections or juvenile justice funds",
        "SEF": "State education agency funds",
        "SF": "Cash or self-payment",
        "SI": "State-financed health insurance plan other than Medicaid",
        "SMHA": "State mental health agency funds",
        "SWFS": "State welfare or child and family services funds",
        "VAF": "U.S. Department of VA funds",
    },
    "payment_assistance": {
        "PA": "Payment assistance available",
        "SS": "Sliding fee scale",
    },
    "special_programs_groups": {
        "TAY": "Young adults",
        "SE": "Seniors or older adults",
        "GL": "LGBTQ",
        "VET": "Veterans",
        "ADM": "Active duty military",
        "MF": "Members of military families",
        "CJ": "Criminal justice / forensic clients",
        "CO": "Clients with co-occurring mental and substance use disorders",
        "HV": "Clients with HIV or AIDS",
        "DV": "Clients with intimate partner violence or domestic violence experience",
        "TRMA": "Clients with trauma",
        "TBI": "Persons with traumatic brain injury",
        "ALZ": "Persons with Alzheimer's or dementia",
        "PED": "Persons with eating disorders",
        "PEFP": "Persons experiencing first-episode psychosis",
        "PTSD": "Persons with post-traumatic stress disorder",
        "SED": "Children/adolescents with serious emotional disturbance",
        "SMI": "Persons 18 and older with serious mental illness",
    },
    "assessment_pretreatment": {
        "STU": "Screening for tobacco use",
    },
    "testing": {
        "HIVT": "HIV testing",
        "STDT": "STD testing",
        "TBS": "TB screening",
        "MST": "Metabolic syndrome monitoring",
        "HBT": "Testing for Hepatitis B",
        "HCT": "Testing for Hepatitis C",
        "LABT": "Laboratory testing",
    },
    "recovery_support": {
        "HS": "Housing services",
        "PEER": "Mentoring / peer support",
    },
    "education_counseling": {
        "TCC": "Smoking / vaping / tobacco cessation counseling",
    },
    "smoking_policy": {
        "SMON": "Smoking not permitted",
        "SMOP": "Smoking permitted without restriction",
        "SMPD": "Smoking permitted in designated area",
    },
    "age_groups_accepted": {
        "CHLD": "Children / adolescents",
        "YAD": "Young adults",
        "ADLT": "Adults",
        "SNR": "Seniors",
    },
    "language_services": {
        "SP": "Spanish",
        "AH": "Sign language services for the deaf and hard of hearing",
        "NX": "American Indian or Alaska Native languages",
        "FX": "Other languages excluding Spanish",
    },
    "vaping_policy": {
        "VAPN": "Vaping not permitted",
        "VAPP": "Vaping permitted without restriction",
        "VPPD": "Vaping permitted in designated area",
    },
    "ancillary_services": {
        "ACT": "Assertive community treatment",
        "AOT": "Assisted Outpatient Treatment",
        "CDM": "Chronic disease / illness management",
        "COOT": "Court-ordered outpatient treatment",
        "DEC": "Diet and exercise counseling",
        "FPSY": "Family psychoeducation",
        "ICM": "Intensive case management",
        "IMR": "Illness management and recovery",
        "LAD": "Legal advocacy",
        "PRS": "Psychosocial rehabilitation services",
        "SEMP": "Supported employment",
        "SH": "Supported housing",
        "TPC": "Therapeutic foster care",
        "VRS": "Vocational rehabilitation services",
        "CM": "Case management service",
        "IPC": "Integrated primary care services",
        "SPS": "Suicide prevention services",
        "ES": "Education services",
    },
}

# Helper function to extract and map codes for a given category
def get_category_descriptions(service_info, category_dict):
    if pd.isna(service_info) or not isinstance(service_info, str):
        return None

    found_descriptions = []
    for code, description in category_dict.items():
        # Use regex for whole word match to avoid false positives (e.g., 'OP' in 'DROPE')
        if re.search(r'\b' + re.escape(code) + r'\b', service_info):
            found_descriptions.append(description)

    return ', '.join(sorted(found_descriptions)) if found_descriptions else None

# Combine all decoded descriptions into a single 'service_code_info_decoded' column
def get_all_category_descriptions(service_info, service_maps):
    if pd.isna(service_info) or not isinstance(service_info, str):
        return None
    all_descriptions = []
    for category_dict in service_maps.values():
        for code, description in category_dict.items():
            if re.search(r'\b' + re.escape(code) + r'\b', service_info):
                all_descriptions.append(description)
    return ' | '.join(sorted(list(set(all_descriptions)))) if all_descriptions else None

df_samhsa['service_code_info_decoded'] = df_samhsa['service_code_info'].apply(
    lambda x: get_all_category_descriptions(x, service_maps)
)

# Create all categorical columns using a loop
for category_name, category_map in service_maps.items():
    # Skip 'service_code_info_decoded' as it's a special combined column
    if category_name != 'service_code_info_decoded':
        df_samhsa[category_name] = df_samhsa['service_code_info'].apply(
            lambda x: get_category_descriptions(x, category_map)
        )

# Display a sample of some of the newly created columns to verify from the merged code
display(df_samhsa[['service_code_info', 'pharmacotherapies', 'treatment_approaches', 'emergency_services', 'payment_funding']].head())

In [ ]:
df_samhsa.drop(['service_code_info', 'service_code_info_decoded'], axis=1, inplace=True)

In [ ]:
selected_row = df_samhsa.iloc[0]

print("All Details for Selected Row (Index 0):")
for i, (column_name, value) in enumerate(selected_row.items(), 1):
    print(f"({i}) {column_name}: {value}")

In [ ]:
df_samhsa.to_csv('samhsa_filtered_data.csv', index=False)
print("DataFrame saved to 'samhsa_filtered_data.csv'")

### DataFrame Column Descriptions

Here's a breakdown of each column in the `df_samhsa` DataFrame:

*   **`city`**: The city where the facility is located.
*   **`state`**: The two-letter state abbreviation where the facility is located (e.g., DE, NJ, NY, PA).
*   **`zip`**: The five-digit ZIP code of the facility.
*   **`phone`**: The main phone number for the facility.
*   **`intake1`**: The primary intake phone number for the facility.
*   **`intake2`**: A secondary intake phone number for the facility.
*   **`intake1a`**: An alternative primary intake phone number, if available.
*   **`intake2a`**: An alternative secondary intake phone number, if available.
*   **`facility_name`**: The combined name of the facility, derived from `name1` and `name2`.
*   **`address`**: The complete street address of the facility, combining `street1` and `street2`.
*   **`type_of_care`**: Decoded descriptions of the types of care provided (e.g., Mental health treatment, Substance use treatment).
*   **`service_setting`**: Decoded descriptions of the settings in which services are provided (e.g., Outpatient, Partial hospitalization / day treatment).
*   **`facility_type`**: Decoded descriptions of the type of facility (e.g., Community mental health center, Psychiatric hospital).
*   **`pharmacotherapies`**: Decoded descriptions of available pharmacotherapies (e.g., Chlorpromazine, Aripiprazole, Nicotine replacement).
*   **`treatment_approaches`**: Decoded descriptions of therapeutic approaches offered (e.g., Cognitive behavioral therapy, Dialectical behavior therapy, Telemedicine / telehealth therapy).
*   **`emergency_services`**: Decoded descriptions of emergency services available (e.g., Crisis intervention team, Psychiatric emergency onsite services).
*   **`facility_operation`**: Decoded descriptions of how the facility is operated (e.g., Private for-profit organization, State government).
*   **`license_certification`**: Decoded descriptions of licenses or certifications held by the facility (e.g., Federally Qualified Health Center, Mental health clinic).
*   **`payment_funding`**: Decoded descriptions of accepted payment methods or funding sources (e.g., Medicare, Medicaid, Private health insurance, Cash or self-payment).
*   **`payment_assistance`**: Decoded descriptions of payment assistance options (e.g., Payment assistance available, Sliding fee scale).
*   **`special_programs_groups`**: Decoded descriptions of special programs or groups served (e.g., Young adults, Veterans, Clients with co-occurring mental and substance use disorders).
*   **`assessment_pretreatment`**: Decoded descriptions of assessment or pretreatment services (e.g., Screening for tobacco use).
*   **`testing`**: Decoded descriptions of testing services available (e.g., HIV testing, Laboratory testing).
*   **`recovery_support`**: Decoded descriptions of recovery support services (e.g., Housing services, Mentoring / peer support).
*   **`education_counseling`**: Decoded descriptions of education and counseling services (e.g., Smoking / vaping / tobacco cessation counseling).
*   **`smoking_policy`**: Decoded description of the facility's smoking policy (e.g., Smoking not permitted, Smoking permitted in designated area).
*   **`age_groups_accepted`**: Decoded descriptions of the age groups accepted by the facility (e.g., Children / adolescents, Adults, Seniors).
*   **`language_services`**: Decoded descriptions of language services provided (e.g., Spanish, Sign language services).
*   **`vaping_policy`**: Decoded description of the facility's vaping policy (e.g., Vaping not permitted, Vaping permitted in designated area).
*   **`ancillary_services`**: Decoded descriptions of additional ancillary services offered (e.g., Case management service, Assertive community treatment, Integrated primary care services).

In [ ]:
df_samhsa.info()